## 依赖导入

本 notebook 汇总综合评价方法：指标数据的归一化/标准化预处理、Logistic 增长型隶属函数标定、人岗匹配的 0-1 整数规划（`cvxpy`）、灰色关联分析、TOPSIS 熵权与秩和比（`scipy.stats.rankdata`）、模糊集合贴近度、隶属度函数加权综合以及 DEA 数据包络分析。统一引入 `cvxpy`、`numpy`、`pandas`、`matplotlib` 与 `scipy`。

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import fsolve
from scipy.stats import rankdata

## 12.1  指标数据的归一化与标准化预处理

读取 `data/14_1_1.txt` 的指标矩阵 a，逐列计算 2 范数 b、列最大值 m1、列最小值 m2，据此生成三种规范化矩阵：R1 = a/b 为列向量归一化，R2 = a/m1 为比例变换，R3 = (a-m2)/(m1-m2) 为极差变换。第 4 列为成本型（越小越好）指标，故分别取 1-a/b、m2/a、(m1-a)/(m1-m2) 作逆向处理。结果 R1 以 `%.4f` 写入 `data/14_1_2.txt`，三种矩阵再写入 `data/14_1_3.xlsx` 的三个 Sheet 便于后续制表。（源程序 ex14_1.py）

In [ ]:
a = np.loadtxt("data/14_1_1.txt")
b = np.linalg.norm(a, axis=0)
m1 = a.max(axis=0)
m2 = a.min(axis=0)
R1 = a / b
R2 = a / m1
R3 = (a - m2) / (m1 - m2)
R1[:, 3] = 1 - a[:, 3] / b[3]
R2[:, 3] = m2[3] / a[:, 3]
R3[:, 3] = (m1[3] - a[:, 3]) / (m1[3] - m2[3])
np.savetxt("data/14_1_2.txt", R1, fmt="%.4f")
f = pd.ExcelWriter("data/14_1_3.xlsx")
pd.DataFrame(R1).to_excel(f, index=None)
pd.DataFrame(R2).to_excel(f, "Sheet2", index=None)
pd.DataFrame(R3).to_excel(f, "Sheet3", index=None)
f.save()

综合测评隶属度标定示例（源程序 anli14_1_1.py）。用 `fsolve` 解出 Logistic 型隶属函数的待定参数：
对增长型部分 `f1` 由 f(1)=0.01、f(3)=0.8 定出 c1（alpha、beta）；对饱和型部分 `f2` 由对数直线在 x=3、x=5 处取值 0.8、1 定出 c2（a、b）。`fx(x)` 将两段按 x∈[1,3]、x∈(3,5] 拼接成分段隶属函数，并绘出曲线（x0=np.linspace(1,5,100)）。
随后读 `data/14_1_1.txt`：第 0 列为初试成绩 d1，第 3 列起为复试指标 d2，经 `fx` 打分后按行取均值再极差标准化得复试成绩 e2，初试成绩同样标准化得 e1，综合得分 f=(e1+e2)/2；`np.argsort(-f)` 从大到小取逆序、再映射为 1..n 的名次，打印得分与名次并把 f 写入 `data/14_1_2.txt`。（原代码中 `f2=fx(2)`、`f4=fx(4)` 为演示标定值的死变量，已删。）

In [ ]:
def f1(t):
    return [
        1 / (1 + t[0] / (1 - t[1]) ** 2) - 0.01,
        1 / (1 + t[0] / (3 - t[1]) ** 2) - 0.8,
    ]


c1 = fsolve(f1, [0.5, 0.5])


def f2(t):
    return [t[0] * np.log(3) + t[1] - 0.8, t[0] * np.log(5) + t[1] - 1]


c2 = fsolve(f2, [0.5, 0.5])


def fx(x):
    return 1 / (1 + c1[0] / (x - c1[1]) ** 2) * ((x >= 1) & (x <= 3)) + (
        c2[0] * np.log(x) + c2[1]
    ) * ((x > 3) & (x <= 5))


x0 = np.linspace(1, 5, 100)
fig, ax = plt.subplots()
ax.plot(x0, fx(x0))

d0 = np.loadtxt("data/14_1_1.txt")
d1 = d0[:, 0]
d2 = d0[:, 3:]
e20 = fx(d2)
e21 = e20.mean(1)
e2 = (e21 - min(e21)) / (max(e21) - min(e21))
e1 = (d1 - min(d1)) / (max(d1) - min(d1))
f = (e1 + e2) / 2
ind = np.argsort(-f)
ind[ind] = np.arange(1, len(ind) + 1)
print("综合得分：\n", f)
print("综合得分排序：\n", ind)
np.savetxt("data/14_1_2.txt", f)
plt.show()

人岗匹配 0-1 整数规划（源程序 anli14_1_2.py）。方法与上一格类似，但增长段端点改为 f(1)=0.01、f(4)=0.8，饱和段对数直线过 (4,0.8)、(7,1)，得隶属函数 f(x)。`g(x)` 把评分差值映射为等级分（差 0→4、-1→5、…、≥3→1），对每位应聘者 i 与每个部门 j，用 `f(g(b_j-a_i)).mean()` 得到部门对应聘者的评分矩阵 s。
读取综合得分 `data/14_1_2.txt` 为 d。用 `cvxpy` 构造 0-1 整数变量 x（n 名应聘者 × m 个部门），目标 `Maximize` 综合得分与匹配评分之和，约束：总录用 8 人、每人至多 1 部门、每部门 1~2 人。以 `GLPK_MI` 求解后打印最优值与最优解，提取非零匹配 `np.nonzero`，按部门序号排序后与评分矩阵 s 一起写入 `data/14_1_4.xlsx`。（原代码中 `f17=f(np.arange(1,8))` 为演示死变量，已删。）

In [ ]:
def f1(t):
    return [
        1 / (1 + t[0] / (1 - t[1]) ** 2) - 0.01,
        1 / (1 + t[0] / (4 - t[1]) ** 2) - 0.8,
    ]


c1 = fsolve(f1, [0.5, 0.5])


def f2(t):
    return [t[0] * np.log(4) + t[1] - 0.8, t[0] * np.log(7) + t[1] - 1]


c2 = fsolve(f2, [0.5, 0.5])


def f(x):
    return 1 / (1 + c1[0] / (x - c1[1]) ** 2) * ((x >= 1) & (x <= 4)) + (
        c2[0] * np.log(x) + c2[1]
    ) * ((x > 4) & (x <= 7))


d1 = np.loadtxt("data/14_1_1.txt")
d2 = np.loadtxt("data/14_1_3.txt")
a = d1[:, 3:]
b = d2[:, 6:]


def g(x):
    return (
        4 * (x == 0)
        + 5 * (x == -1)
        + 6 * (x == -2)
        + 7 * (x == -3)
        + 3 * (x == 1)
        + 2 * (x == 2)
        + (x >= 3)
    )


m = b.shape[0]
n = a.shape[0]
s = np.zeros((n, m))
for i in range(n):
    for j in range(m):
        t1 = g(b[j, :] - a[i, :])
        t2 = f(t1)
        s[i, j] = t2.mean()
d = np.loadtxt("data/14_1_2.txt")

x = cp.Variable((n, m), integer=True)
obj = cp.Maximize(
    cp.sum(cp.multiply(d, cp.sum(x, axis=1))) + cp.sum(cp.multiply(s, x))
)
con = [
    cp.sum(x) == 8,
    cp.sum(x, axis=1) <= 1,
    cp.sum(x, axis=0) >= 1,
    cp.sum(x, axis=0) <= 2,
    x >= 0,
    x <= 1,
]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值：", prob.value)
print("最优解：\n", x.value)
i, j = np.nonzero(x.value)
pf0 = s * x.value
pf = pf0[np.nonzero(pf0)]
out = np.vstack([j + 1, i + 1, d[i], pf])
ind = np.argsort(j)
out = out[:, ind]
fid = pd.ExcelWriter("data/14_1_4.xlsx")
pd.DataFrame(s).to_excel(fid, index=None)
pd.DataFrame(out).to_excel(fid, "Sheet2", index=None)
fid.save()

基于相互满意度的人岗匹配（源程序 anli14_1_3.py）。`fx(x)=log(4-x)/log(3)` 在 x=1,2,3 处取值得到权重向量 w，用于部门满意度的分档加权。读取 `data/14_1_1.txt` 的志愿类别列 a（第 1、2 列）与 `data/14_1_3.txt` 的部门客观评分列，`g(x)` 把志愿类别映射到部门编号。构造权重矩阵 wij（第一志愿权重 1、第二志愿权重 w[1]），与部门客观均值相乘得应聘者对部门的满意度 t；读取上一步结果 `data/14_1_4.xlsx` 得部门对应聘者的评分 s，相互满意度 `r=sqrt(s*t)`。再用 `cvxpy` 求与上一格相同约束的 0-1 最大匹配，打印最优值/最优解、提取非零满意度并按部门排序，写入 `data/14_1_5.xlsx`。

In [ ]:
def fx(x):
    return np.log(4 - x) / np.log(3)


w = fx(np.arange(1, 4))
d1 = np.loadtxt("data/14_1_1.txt")
d2 = np.loadtxt("data/14_1_3.txt")
a = d1[:, [1, 2]]


def g(x):
    return (
        0 * (x == 1)
        + np.array([1, 2]) * (x == 2)
        + np.array([3, 4]) * (x == 3)
        + np.array([5, 6]) * (x == 4)
    )


wij = np.zeros((16, 7))
for i in range(16):
    wij[i, g(a[i, 0])] = 1
    wij[i, g(a[i, 1])] = w[1]
tj = d2[:, 1:6]
t = wij * tj.mean(axis=1)
s = pd.read_excel("data/14_1_4.xlsx").values
r = np.sqrt(s * t)

x = cp.Variable((16, 7), integer=True)
obj = cp.Maximize(cp.sum(cp.multiply(r, x)))
con = [
    cp.sum(x) == 8,
    cp.sum(x, axis=1) <= 1,
    cp.sum(x, axis=0) >= 1,
    cp.sum(x, axis=0) <= 2,
    x >= 0,
    x <= 1,
]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值：", prob.value)
xx = x.value
print("最优解：\n", xx)
i, j = np.nonzero(xx)
fc0 = r * xx
fc = fc0[np.nonzero(fc0)]
out = np.vstack([j + 1, i + 1, fc])
ind = np.argsort(j)
out = out[:, ind]
pd.DataFrame(out).to_excel("data/14_1_5.xlsx", index=None)

## 12.2  TOPSIS、灰色关联、熵权与秩和比综合评价

读取 `data/14_1_2.txt` 的规范化矩阵 a，用四种方法分别给对象打分：①TOPSIS——以列最大/最小为正、负理想解 bp/bm，算欧氏距离 d1、d2，贴近度 f1=d2/(d1+d2)；②灰色关联——参考序列与各行之差 c，取最大差 m1、最小差 m2，分辨系数 r=0.5，关联系数 xs 后按行求均值 f2；③熵权法——特征比重 P=a/a.sum(axis=0)，信息熵 e 求出差异系数 g=1-e，权重 `w=g/g.sum()`（g 为 ndarray，`.sum()` 与原 `sum(g)` 完全等价），评价值 f3=P@w；④秩和比——`rankdata(a,axis=0)` 逐列编秩后取行均值除以对象数 n 得 RSR。分别打印四类评价值（灰色关联、熵权、秩和比保留 4 位）。（源程序 ex14_2.py）

In [ ]:
a = np.loadtxt("data/14_1_2.txt")
bp = a.max(axis=0)
bm = a.min(axis=0)
d1 = np.linalg.norm(a - bp, axis=1)
d2 = np.linalg.norm(a - bm, axis=1)
f1 = d2 / (d1 + d2)
print("TOPSIS 评价值：", f1)

c = bp - a
m1 = c.max()
m2 = c.min()
r = 0.5
xs = (m2 + r * m1) / (c + r * m1)
f2 = xs.mean(axis=1)
print("灰色关联度：", np.round(f2, 4))

n = a.shape[0]
s = a.sum(axis=0)
P = a / s
e = -(P * np.log(P)).sum(axis=0) / np.log(n)
g = 1 - e
w = g / g.sum()
f3 = P @ w
print("评价值：", np.round(f3, 4))

R = rankdata(a, axis=0)
RSR = R.mean(axis=1) / n
print("秩和比：", np.round(RSR, 4))

## 12.3  模糊集合的贴近度

给定三个隶属函数向量 a 与目标向量 b，对每个 e 计算贴近度 $1-\sum|e-b|/3$（$\sum|e-b|$ 即 `abs(e-b).sum()`，与内置 `sum` 等价），结果保留 4 位打印。原逐行 append 循环已改写为列表推导式，数值不变。（源程序 ex14_3.py）

In [ ]:
a = np.array([[0.4, 0.3, 0.5, 0.3], [0.3, 0.3, 0.4, 0.4], [0.2, 0.3, 0.3, 0.3]])
b = np.array([0.2, 0.3, 0.4, 0.3])
N = np.array([1 - abs(e - b).sum() / 3 for e in a])
print("贴近度为：", np.round(N, 4))

## 12.4  隶属度函数加权综合评值

读取 `data/14_4.txt`（5 行指标），为第 i 行定义隶属度函数 f1..f5（分别为线性型、负线性型、梯形/降半型等）。原代码用 `eval("f{i+1}(a[i])")` 动态调用，存在安全隐患（S307），现改为安全等价写法：把函数放入列表 `funcs=[f1,f2,f3,f4,f5]`，用 `R=np.array([funcs[i](a[i]) for i in range(len(a))])` 逐行调用对应函数——已用 kaggle 环境断言新旧结果数组完全一致（byte-identical）。再以权重 w 求综合评价值 B=w@R，保留 4 位打印。（源程序 ex14_4.py）

In [ ]:
a = np.loadtxt("data/14_4.txt")


def f1(x):
    return x / 8800


def f2(x):
    return 1 - x / 8000


def f3(x):
    return (x <= 5.5) + (8 - x) / (8 - 5.5) * ((x > 5.5) & (x < 8))


def f4(x):
    return 1 - x / 200


def f5(x):
    return (x - 50) / (1500 - 50)


funcs = [f1, f2, f3, f4, f5]
R = np.array([funcs[i](a[i]) for i in range(len(a))])
w = np.array([0.25, 0.2, 0.2, 0.1, 0.25])
B = w @ R
print("评价值：", np.round(B, 4))

## 12.5  DEA 数据包络分析

读取 `data/14_5.txt`，前三列为投入 a、后两列为产出 b。对每个决策单元 j，用 `cvxpy` 求权重变量 u（投入权，pos=True）、v（产出权，pos=True）：约束所有单元“产出加权 ≥ 投入加权”且第 j 个单元投入加权=1，目标最大化第 j 个单元的产出加权。以 `GLPK_MI` 求解后打印各单元最优值与最优权重（各保留 4 位）。（源程序 ex14_5.py）

In [ ]:
d = np.loadtxt("data/14_5.txt")
a = d[:, :3]
b = d[:, 3:]
u = cp.Variable(3, pos=True)
v = cp.Variable(2, pos=True)
for j in range(10):
    con = [a @ u >= b @ v, a[j] @ u == 1]
    obj = cp.Maximize(b[j] @ v)
    prob = cp.Problem(obj, con)
    prob.solve(solver="GLPK_MI")
    print("第", str(j + 1), "个对象最优值：", round(prob.value, 4))
    print("最优解：\n", np.round(u.value, 4), "\n", np.round(v.value, 4))